<a href="https://colab.research.google.com/github/the-mayankjha/pihu_wakeup/blob/main/PihuWakeWord.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PiHu WakeWord Detection
## Setting up Environment


In [4]:
# 1. Clone openwakeword & the correct piper-sample-generator fork
!git clone https://github.com/dscripka/openwakeword
!git clone https://github.com/dscripka/piper-sample-generator

# 2. Download the REQUIRED high-quality Piper model
!wget -q -O piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'

# 3. Install system packages
!apt-get update -qq
!apt-get install -y espeak-ng

# 4. Install all Python dependencies (without TensorFlow!)
import sys
!{sys.executable} -m pip install -e ./openwakeword
!{sys.executable} -m pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14
!{sys.executable} -m pip install audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!{sys.executable} -m pip install pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19
!{sys.executable} -m pip install espeak-phonemizer piper-phonemize-cross

# 5. Apply PyTorch 2.x Bug Fixes using Python
import os

# Fix 1: Remove deprecated torchaudio.set_audio_backend
io_file = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"
if os.path.exists(io_file):
    with open(io_file, "r") as f:
        content = f.read()
    with open(io_file, "w") as f:
        f.write(content.replace('torchaudio.set_audio_backend("soundfile")', '# torchaudio.set_audio_backend("soundfile")'))

# Fix 2: Add weights_only=False to torch.load in piper generator
gen_file = "/content/piper-sample-generator/generate_samples.py"
if os.path.exists(gen_file):
    with open(gen_file, "r") as f:
        content = f.read()
    with open(gen_file, "w") as f:
        f.write(content.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)'))

print("Pihu Env Setup /n")
print("Environment setup and patching complete!")

fatal: destination path 'openwakeword' already exists and is not an empty directory.
fatal: destination path 'piper-sample-generator' already exists and is not an empty directory.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 18 not upgraded.
Obtaining file:///content/openwakeword
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done
  Created wheel for openwakeword: filename=openwakeword-0.6.0-0.editable-py3-none-any.whl size=

## Download Background Noise & Room Environments


In [6]:
import os
import numpy as np
import scipy
from tqdm import tqdm
from pathlib import Path
import datasets

# Ensure we are in the root directory
%cd /content

# 1. Download Room Impulse Responses (MIT)
os.makedirs("./mit_rirs", exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="Downloading MIT RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join("./mit_rirs", name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 2. Download Background Noise (AudioSet)
os.makedirs("audioset", exist_ok=True)
!wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
!cd audioset && tar -xf bal_train09.tar

os.makedirs("./audioset_16k", exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="Processing AudioSet"):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join("./audioset_16k", name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 3. Download Background Music (FMA)
os.makedirs("./fma", exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
for i in tqdm(range(3600//30), desc="Downloading FMA Music"):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join("./fma", name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 4. Download Validation Files
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

print("All the assets for Pihu Downloaded")

/content


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


Processing AudioSet: 0it [00:00, ?it/s]


All the assets for Pihu Downloaded


##  Configure "Pihu" Variations

In [7]:
import yaml
import sys

%cd /content

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Your exhaustive list of Indian phonetic pronunciations
config["target_phrase"] = [
    "pihu", "peehoo", "pee hoo", "peewhoo", "pee who", "pee whoo",
    "pea hoo", "peahoo", "pea who", "peawho", "pea whoo", "pi hoo", "pi who",
    "hey pihu", "hey peehoo", "hey pee hoo", "hey peewhoo", "hey pee who",
    "hey pea hoo", "hey pea who", "hay pee hoo", "hay peewhoo", "hay pee who",
    "hi pihu", "hi peehoo", "hi pee hoo", "hi peewhoo", "hi pee who",
    "hi pea hoo", "hi pea who", "high pee hoo", "high peewhoo", "high pee who"
]

config["model_name"] = "pihu"
config["n_samples"] = 5000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("Configuration saved to my_model.yaml")

/content
Configuration saved to my_model.yaml


In [13]:
import os

# Patch deep-phonemizer for PyTorch 2.6
dp_file = "/usr/local/lib/python3.12/dist-packages/dp/model/model.py"
if os.path.exists(dp_file):
    with open(dp_file, "r") as f:
        content = f.read()
    with open(dp_file, "w") as f:
        # Inject the weights_only=False flag into the torch.load command
        f.write(content.replace(
            'checkpoint = torch.load(checkpoint_path, map_location=device)',
            'checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)'
        ))

print("deep-phonemizer patched!")

deep-phonemizer patched!


## Train & Save the Model

In [10]:
import sys
!{sys.executable} -m pip install webrtcvad onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached onnx-1.21.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 78.7 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73513 sha256=2ac5572474b4b21c7c6648c65f281bacb7ea14c5bcabdcbb41334d465309b4bc
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
Successfully built webrtcvad


In [ ]:
# 1. Generate synthetic clips (Will take a while)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips
# 2. Augment clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips
# 3. Train model (this step skips TFLite conversion automatically!)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model
print("TRAINING COMPLETE! Your 'pihu.onnx' model is ready to download from the 'my_custom_model' folder.")

INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
/usr/local/lib/python3.12/dist-packages/dp/model/model.py:69: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(encoder_layer=encoder_layer,
DEBUG:dp.phonemizer:Initializing phonemizer with model step 1120000
DEBUG:dp.phonemizer:Initializing phonemizer with model step 1120000
DEBUG:dp.phonemizer:Initializing phonemizer with model step 1120000
DEBUG:dp.phonemizer:Initializin